In [ ]:
!pip install jupyter-tensorboard

In [ ]:
import torch
import os
import sys
import json
import tensorboard
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
from accelerate import PartialState

In [ ]:
#model path
mpath = "/workspace/Llama-2-7b-chat-hf"
#gpu
# device_map = {'':1}
device_map={"": 0}
#dataset path
dataset_path = "clean_meteo.csv"
#new model path
new_model = "Llama-2-7b-ovi-fn"

# Load the dataset
dataset = load_dataset('csv', data_files=dataset_path, split='train')
# Visualize
dataset


In [ ]:
# Process dataset
DEFAULT_SYSTEM_PROMPT = """
Below is a JSON of weather forecast details. Write a summary based on these values.
""".strip()
def generate_training_prompt(
    json_input: str, output: str, system_prompt: str = DEFAULT_SYSTEM_PROMPT
) -> str:
    return f"""<s>[INST] <<SYS>>
{system_prompt}
<</SYS>>


{json_input.strip()} [/INST] {output} </s>""".strip()

def generate_text(data_point):
    json_input = data_point['Input']
 
    output = data_point['Output']
    return {
        "Input": json_input,
        "Output": output,
        "text": generate_training_prompt(json_input, output),
    }

 
def process_dataset(data: Dataset):
    return (
        data.shuffle(seed=42)
        .map(generate_text)
    )
dataset = process_dataset(dataset)
dataset

In [ ]:
#check how it looks
dataset[0]['text']

In [ ]:
#QLora params

lora_r = 64

lora_alpha = 16

lora_dropout = 0.1

#BnB params

use_4bit = True

bnb_4bit_compute_dtype = "float16"

bnb_4bit_quant_type = "nf4"

use_nested_quant = False


#Training Arguments params

output_dir = "./results"

num_train_epochs = 1

fp16 = False
bf16 = False

per_device_train_batch_size = 4

per_device_eval_batch_size = 4

gradient_accumulation_steps = 1

gradient_checkpointing = True

max_grad_norm = 0.3

learning_rate = 2e-4

weight_decay = 0.001

optim = "paged_adamw_32bit"

lr_scheduler_type = "cosine"

max_steps = 200

warmup_ratio = 0.03

group_by_length = True

save_steps = 0

logging_steps = 25

# SFT params


max_seq_length = None

packing = False

In [ ]:
# Load tokenizer and model with QLoRA configuration
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Check GPU compatibility with bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    mpath,
    quantization_config=bnb_config,
    device_map=device_map
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(mpath, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix weird overflow issue with fp16 training

# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)

# Set training parameters
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard"
)

# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=packing,
)


In [ ]:
# Train model
trainer.train()
# Save trained model
trainer.model.save_pretrained(new_model)

In [ ]:
# import os
# import tensorflow as tf
# from tensorboard import notebook

# # Specify the path to the tensorboard executable
# tensorboard_path = '/home/ovidiug/.local/lib/python3.11/site-packages/tensorboard/main.py'
# os.environ['TENSORBOARD_BINARY'] = tensorboard_path

# # Specify the log directory where your TensorBoard logs are stored
# log_dir = 'results/runs'

# # Start TensorBoard
# notebook.start("--logdir {}".format(log_dir))
# Start TensorBoard
# !python3 /home/ovidiug/.local/lib/python3.11/site-packages/tensorboard/main.py --logdir=results/runs

In [ ]:
logging.set_verbosity(logging.CRITICAL)
# Run text generation pipeline with our next model
prompt2 = {
    "time": ["0400", "0700", "1000", "1300", "1600", "1900", "2200", "0100"],
    "temperature_real (°C)": [10, 8, 15, 20, 18, 16, 17, 17],
    "temperature_felt (°C)": [7, 6, 12, 18, 15, 12, 13, 13],
    "wind_speed (km/h)": ["5-7", "3-5", "5-10", "18-33", "18-55", "23-56", "22-57", "23-56"],
    "wind_direction": ["N", "NW", "ESE", "SSW", "SSW", "SSW", "SSW", "S"],
    "precipitation_size (mm/3h)": ["-", "-", "-", "-", "-", "-", "-", "-"],
    "precipitation_probability": ["0%", "0%", "0%", "0%", "0%", "0%", "0%", "10%"],
    "cloud_cover": ["Clear, cloudless sky", "Clear with few low clouds", "Clear with few low clouds and cirrus", "Clear with few low clouds and cirrus", "Clear with few low clouds and cirrus", "Clear with few low clouds and few cirrus", "Partly cloudy and cirrus", "Partly cloudy and cirrus"]
}


pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=600)
result = pipe(f"<s>[INST] {prompt2} [/INST]")
print(result[0]['generated_text'])


In [ ]:
# Empty VRAM
del model
del pipe
del trainer
import gc
gc.collect()
gc.collect()

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    mpath,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map=device_map,
)
model = PeftModel.from_pretrained(base_model, new_model)
model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(mpath, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
logging.set_verbosity(logging.CRITICAL)

prompt = {'time': ['0300', '0600', '0900', '1200', '1500', '1800', '2100', '0000'], 'temperature_real (°C)': [8, 7, 13, 19, 17, 15, 16, 16], 'temperature_felt (°C)': [5, 4, 10, 16, 14, 10, 11, 11], 'wind_speed (km/h)': ['6-8', '4-6', '6-13', '19-35', '19-57', '24-58', '23-59', '24-58'], 'wind_direction': ['NW', 'W', 'SSE', 'SSW', 'SSW', 'SSW', 'SSW', 'S'], 'precipitation_size (mm/3h)': ['-', '-', '-', '-', '-', '-', '-', '-'], 'precipitation_probability': ['0%', '0%', '0%', '0%', '0%', '0%', '0%', '20%'], 'cloud_cover': ['Clear, cloudless sky', 'Clear with few low clouds', 'Clear with few low clouds and cirrus', 'Clear with few low clouds and cirrus', 'Clear with few low clouds and cirrus', 'Clear with few low clouds and few cirrus', 'Partly cloudy and cirrus', 'Partly cloudy and cirrus']}


pipe = pipeline(task="text-generation", model=base_model, tokenizer=tokenizer, max_length=1000)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print("Base model:")
print(result[0]['generated_text'])

pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=1000)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print("New model:")
print(result[0]['generated_text'])
